# GRAPH NN AS SHOWN DURING LECTURES

In [ ]:
# ==============================================================================
# IMPORTS AND SETUP
# ==============================================================================
import networkx as nx
import numpy as np
import torch
from torch.nn.parameter import Parameter
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ==============================================================================
# PART 1: THE DATA (GRAPH REPRESENTATION)
# ==============================================================================
# A graph consists of Nodes (entities) and Edges (connections).
G = nx.Graph()
blue, orange, green = "#1f77b4", "#ff7f0e", "#2ca02c"

# We add 4 nodes. Each node gets a feature: a "color" attribute.
G.add_nodes_from([(1, {"color": blue}),
                  (2, {"color": orange}),
                  (3, {"color": blue}),
                  (4, {"color": green})])

# We add edges. In an undirected graph, an edge (1,2) means 1 is connected
# to 2, and 2 is connected to 1. This defines the "neighborhoods".
G.add_edges_from([(2, 1), (2, 3), (1, 3), (3, 4)])

# --- THE ADJACENCY MATRIX (A) ---
# This is a square NxN matrix (4x4 here).
# If A[i, j] = 1, it means node i and node j are connected.
# In a GNN, 'A' acts as the router: it dictates how nodes share information.
A = np.asarray(nx.adjacency_matrix(G).todense())

# --- THE FEATURE MATRIX (X) ---
# We convert our node colors into numerical representations (one-hot vectors).
def build_graph_color_label_representation(G, mapping_dict):
    one_hot_idxs = np.array([mapping_dict[v] for v in nx.get_node_attributes(G, 'color').values()])
    one_hot_encoding = np.zeros((one_hot_idxs.size, len(mapping_dict)))
    one_hot_encoding[np.arange(one_hot_idxs.size), one_hot_idxs] = 1
    return one_hot_encoding

# X is an NxF matrix (4 nodes x 3 color features).
X = build_graph_color_label_representation(G, {green: 0, blue: 1, orange: 2})

# ==============================================================================
# PART 2: THE MATH OF A GRAPH CONVOLUTION (NUMPY)
# ==============================================================================
f_in = X.shape[1] # Input features (3)
f_out = 6         # Output embedding dimension

# A GNN layer learns two sets of weights:
W_1 = np.random.rand(f_in, f_out) # W_1: How a node transforms its OWN features.
W_2 = np.random.rand(f_in, f_out) # W_2: How a node transforms its NEIGHBORS' features.

# --- MESSAGE PASSING EQUATION: H = X*W_1 + A*X*W_2 ---
# 1. np.dot(X, W_1): Every node applies a transformation to itself.
# 2. np.dot(A, X): The magic step. Multiplying A by X mathematically SUMS UP
#    the features of all adjacent neighbors for every node simultaneously.
# 3. np.dot(np.dot(A, X), W_2): Apply the W_2 transformation to those summed neighbors.
h = np.dot(X, W_1) + np.dot(np.dot(A, X), W_2)

# ==============================================================================
# PART 3: THE PYTORCH IMPLEMENTATION
# ==============================================================================
# Here we turn the Numpy math above into a trainable neural network layer.
class BasicGraphConvolutionLayer(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # We define W1, W2, and bias as learnable parameters.
        # Backpropagation will adjust these to minimize our loss later.
        self.W1 = Parameter(torch.rand((in_channels, out_channels), dtype=torch.float32))
        self.W2 = Parameter(torch.rand((in_channels, out_channels), dtype=torch.float32))
        self.bias = Parameter(torch.zeros(out_channels, dtype=torch.float32))

    def forward(self, X, A):
        # Step 1: Prepare messages from neighbors (X * W2)
        potential_msgs = torch.mm(X, self.W2)

        # Step 2: Route the messages using the Adjacency matrix (A * potential_msgs)
        propagated_msgs = torch.mm(A, potential_msgs)

        # Step 3: Update own state (X * W1)
        root_update = torch.mm(X, self.W1)

        # Step 4: Combine everything and add bias
        output = propagated_msgs + root_update + self.bias
        return output

# ==============================================================================
# PART 4: BATCHING AND READOUT (THE TRICKY PART)
# ==============================================================================
# Graphs have different numbers of nodes. Standard neural networks hate variable sizes.
# To feed multiple graphs into the network at once, we treat a batch of graphs
# as one GIANT, disconnected graph.

def get_batch_tensor(graph_sizes):
    # This creates a "mask" matrix. It tells the network which nodes belong
    # to which original graph inside the giant batch graph.
    starts = [sum(graph_sizes[:idx]) for idx in range(len(graph_sizes))]
    stops = [starts[idx]+graph_sizes[idx] for idx in range(len(graph_sizes))]
    tot_len = sum(graph_sizes)
    batch_size = len(graph_sizes)

    batch_mat = torch.zeros([batch_size, tot_len]).float()
    for idx, starts_and_stops in enumerate(zip(starts, stops)):
        start, stop = starts_and_stops
        batch_mat[idx, start:stop] = 1 # Mark the nodes for this specific graph
    return batch_mat

def global_sum_pool(X, batch_mat):
    # We want to classify the WHOLE graph, not individual nodes.
    # So we must squish all node embeddings into a single fixed-size vector.
    if batch_mat is None or batch_mat.dim() == 1:
        return torch.sum(X, dim=0).unsqueeze(0) # Just sum all nodes if only 1 graph
    else:
        # Multiply by the batch mask. This ensures we only sum the nodes
        # that belong to the same graph together, keeping batches separate.
        return torch.mm(batch_mat, X)

def collate_graphs(batch):
    # This function builds the giant batch graph.
    adj_mats = [graph['A'] for graph in batch]
    sizes = [A.size(0) for A in adj_mats]
    tot_size = sum(sizes)

    batch_mat = get_batch_tensor(sizes)
    feat_mats = torch.cat([graph['X'] for graph in batch], dim=0) # Stack features
    labels = torch.cat([graph['y'] for graph in batch], dim=0)    # Stack labels

    # We place each graph's Adjacency matrix on the diagonal of a massive matrix.
    # The rest is zeros. This guarantees nodes in Graph 1 cannot pass messages
    # to nodes in Graph 2 during the torch.mm(A, X) step!
    batch_adj = torch.zeros([tot_size, tot_size], dtype=torch.float32)
    accum = 0
    for adj in adj_mats:
        g_size = adj.shape[0]
        batch_adj[accum:accum+g_size, accum:accum+g_size] = adj
        accum += g_size

    return {'A': batch_adj, 'X': feat_mats, 'y': labels, 'batch': batch_mat}

# ==============================================================================
# PART 5: THE FULL NETWORK ARCHITECTURE
# ==============================================================================
class NodeNetwork(torch.nn.Module):
    def __init__(self, input_features):
        super().__init__()
        # Two GNN layers. Because there are two, a node can receive information
        # from its neighbors, AND its neighbors' neighbors (2 hops away).
        self.conv_1 = BasicGraphConvolutionLayer(input_features, 32)
        self.conv_2 = BasicGraphConvolutionLayer(32, 32)

        # Standard Multi-Layer Perceptron (MLP) to process the final graph vector
        self.fc_1 = torch.nn.Linear(32, 16)
        self.out_layer = torch.nn.Linear(16, 2)

    def forward(self, X, A, batch_mat):
        # 1. Message Passing Phase
        x = self.conv_1(X, A).clamp(0) # .clamp(0) acts like a ReLU activation function
        x = self.conv_2(x, A).clamp(0)

        # 2. Readout Phase (Crush N nodes into 1 graph vector)
        output = global_sum_pool(x, batch_mat)

        # 3. Classification Phase
        output = self.fc_1(output)
        output = self.out_layer(output)
        return F.softmax(output, dim=1) # Output probability of belonging to class 0 or 1

# ==============================================================================
# PART 6: TESTING THE IMPLEMENTATION
# ==============================================================================
# Build a helper function to format graphs for PyTorch
def get_graph_dict(G, mapping_dict):
    A = torch.from_numpy(np.asarray(nx.adjacency_matrix(G).todense())).float()
    X = torch.from_numpy(build_graph_color_label_representation(G, mapping_dict)).float()
    y = torch.tensor([[1, 0]]).float() # Dummy label
    return {'A': A, 'X': X, 'y': y, 'batch': None}

# Create a dummy dataset of 4 graphs
mapping_dict = {green: 0, blue: 1, orange: 2}
graph_list = [get_graph_dict(graph, mapping_dict) for graph in [G, G, G, G]] # Just duplicating G for the example

class ExampleDataset(Dataset):
    def __init__(self, graph_list): self.graphs = graph_list
    def __len__(self): return len(self.graphs)
    def __getitem__(self, idx): return self.graphs[idx]

dset = ExampleDataset(graph_list)
# Use our custom collate function to handle the weird batching geometry
loader = DataLoader(dset, batch_size=2, shuffle=False, collate_fn=collate_graphs)

# Run a forward pass!
torch.manual_seed(123)
net = NodeNetwork(node_features=3)

batch_results = []
for b in loader:
    batch_results.append(net(b['X'], b['A'], b['batch']).detach())

print("Network output for a batch of 2 graphs:")
print(batch_results[0])

# REAL GRAPH NN

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# ==========================================
# 1. DATA PREPARATION (The PyG Way)
# ==========================================
# Instead of a dense NxN adjacency matrix filled mostly with zeros (which wastes memory),
# PyG uses an "edge_index" (Coordinate Format).
# It is a 2xE matrix where E is the number of edges.
# The top row is the source node, the bottom row is the destination node.

# Let's recreate a simple 4-node graph: 0-1, 1-2, 2-3, 3-0
edge_index = torch.tensor([
    [0, 1, 2, 3, 1, 2, 3, 0], # Source nodes
    [1, 2, 3, 0, 0, 1, 2, 3]  # Destination nodes (we include both directions)
], dtype=torch.long)

# Node features (X matrix). 4 nodes, each with 3 features (e.g., RGB colors, one-hot encoded)
x = torch.tensor([
    [-1,  0,  1], # Features for Node 0
    [ 0,  1, -1], # Features for Node 1
    [ 1, -1,  0], # Features for Node 2
    [ 0,  0,  1]  # Features for Node 3
], dtype=torch.float)

# Graph label (y). Let's say this whole graph represents a specific molecule, labeled as class '1'.
y = torch.tensor([1], dtype=torch.long)

# Package the features, edges, and labels into a PyG Data object
graph_data = Data(x=x, edge_index=edge_index, y=y)

# To demonstrate batching, let's pretend we have a dataset of 3 identical graphs
dataset = [graph_data, graph_data, graph_data]

# DataLoader handles the complex batching automatically!
# You no longer need to write custom `get_batch_tensor` functions to build massive diagonal matrices.
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# ==========================================
# 2. DEFINING THE GNN MODEL
# ==========================================
class ModernGNN(torch.nn.Module):
    def __init__(self, num_node_features, num_classes):
        super().__init__()

        # GCNConv completely replaces your custom `BasicGraphConvolutionLayer`.
        # It mathematically performs: H = A_hat * X * W
        # but it is highly optimized under the hood for sparse edge lists.
        # It maps our 3 input features to 16 hidden embedding dimensions.
        self.conv1 = GCNConv(num_node_features, 16)
        self.conv2 = GCNConv(16, 16)

        # A standard PyTorch linear layer for the final classification
        self.classifier = torch.nn.Linear(16, num_classes)

    def forward(self, x, edge_index, batch):
        # --- PHASE 1: MESSAGE PASSING ---

        # Layer 1: Nodes look at direct neighbors, gather features, and update themselves.
        x = self.conv1(x, edge_index)
        x = F.relu(x) # Activation function (similar to your .clamp(0))

        # Layer 2: Nodes look at neighbors again. Because their neighbors just updated
        # using THEIR neighbors, nodes now see features from 2 hops away.
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # --- PHASE 2: READOUT (GLOBAL POOLING) ---

        # Right now, `x` is a matrix of node embeddings.
        # But we want to classify the *entire graph*, not individual nodes.
        # global_mean_pool takes all nodes in a graph and averages them into a single vector.
        # The 'batch' variable is a magic PyG vector that tells the function exactly
        # which nodes belong to graph A, graph B, etc., inside the current batch.
        x = global_mean_pool(x, batch)

        # --- PHASE 3: CLASSIFICATION ---

        # Pass the single, pooled graph-level embedding through a standard neural network layer.
        out = self.classifier(x)

        return out

# ==========================================
# 3. THE TRAINING LOOP
# ==========================================
# Initialize the model, optimizer, and loss function
model = ModernGNN(num_node_features=3, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train() # Set PyTorch model to training mode
    total_loss = 0

    for data in loader:         # Iterate through the batches of graphs
        optimizer.zero_grad()   # Clear old gradients from the previous step

        # Forward pass: Feed the features, edges, and the batch mapping to the model
        out = model(data.x, data.edge_index, data.batch)

        # Calculate loss by comparing the network's prediction to the true label (data.y)
        loss = criterion(out, data.y)

        # Backward pass: Compute the mathematical gradients (how to adjust the weights)
        loss.backward()

        # Optimizer step: Actually adjust the model's weights W1, W2, etc.
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# Run a few epochs to see the network "learn"
print("Starting training loop...\n")
for epoch in range(1, 6):
    loss = train()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')